In [1]:
import pandas as pd 
import numpy as np


In [2]:
import pandas as pd
import os

# Use backslashes for Windows paths
data_path = r"/kaggle/input/datasets/pradyumansharmaaa/mercari/train.tsv"

# Check if file exists and is readable
if os.path.isfile(data_path):
    try:
        df = pd.read_csv(data_path, sep="\t")
        print(f"File loaded successfully!")
    except PermissionError:
        print(f"Permission denied. Check file permissions at: {data_path}")
else:
    print(f"File not found: {data_path}")
    print("Check if the path exists or adjust the directory name.")

File loaded successfully!


In [3]:
train = df

In [4]:
train = train[train["price"] > 0].reset_index(drop=True)

print(train.shape)

(1481661, 8)


In [5]:
train.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1481661 entries, 0 to 1481660
Data columns (total 8 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   train_id           1481661 non-null  int64  
 1   name               1481661 non-null  object 
 2   item_condition_id  1481661 non-null  int64  
 3   category_name      1475347 non-null  object 
 4   brand_name         849325 non-null   object 
 5   price              1481661 non-null  float64
 6   shipping           1481661 non-null  int64  
 7   item_description   1481655 non-null  object 
dtypes: float64(1), int64(3), object(4)
memory usage: 637.3 MB


## Train / Test Matrix

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

train["name"] = (
    train["name"].fillna("")
    + " "
    + train["brand_name"].fillna("")
)

tfidf = TfidfVectorizer(
    max_features=100000,
    token_pattern=r"\w+"
)

X_name = tfidf.fit_transform(train["name"])

print(X_name.shape)

(1481661, 100000)


## For Text Field with TF-idf

In [7]:
train["text"] = (
    train["item_description"].fillna("")
    + " "
    + train["name"]
    + " "
    + train["category_name"].fillna("")
)



In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

text_tfidf = TfidfVectorizer(
    max_features=100000,
    token_pattern=r"\w+",
    ngram_range=(1, 2)
)

X_text = text_tfidf.fit_transform(train["text"])

print(X_text.shape)

(1481661, 100000)


## by using Dict vEcotrizer

In [9]:
from sklearn.feature_extraction import DictVectorizer

cat = train[
    ["shipping", "item_condition_id"]
].to_dict("records")

dv = DictVectorizer()

X_cat = dv.fit_transform(cat)

print(X_cat.shape)

(1481661, 2)


## comoobining alll 

In [10]:
from scipy.sparse import hstack

X = hstack([
    X_name,
    X_text,
    X_cat
]).astype("float32")

print(X.shape)

(1481661, 200002)


In [11]:
print("Non-zero entries:", X.nnz)

density = X.nnz / (X.shape[0] * X.shape[1])

print("Density:", density)

#3 if it is not sparse in case it will be around 1.4 tb+ data fk

Non-zero entries: 85352367
Density: 0.00028802645640556496


In [12]:
from sklearn.model_selection import train_test_split
import numpy as np

y = np.log1p(train["price"].values)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.05,
    random_state=42
)

print(X_train.shape)
print(X_valid.shape)

(1407577, 200002)
(74084, 200002)


## MOdel

In [13]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(192, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-3),
    loss="mse"
)

model.summary()

2026-06-23 17:26:21.271635: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782235581.448879      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782235581.498095      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782235581.925001      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782235581.925032      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782235581.925035      23 computation_placer.cc:177] computation placer alr

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 192)            │    38,400,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,417,153 (146.55 MB)

 Trainable params: 38,417,153 (146.55 MB)

 Non-trainable params: 0 (0.00 B)

## learning -> Tensorflow struggles with sparse matrixes so we have to convert to Sparse Tensors

In [14]:
import tensorflow as tf

def scipy_to_tf_sparse(X):
    X = X.tocoo()

    indices = np.vstack((X.row, X.col)).T

    return tf.sparse.SparseTensor(
        indices=indices,
        values=X.data.astype(np.float32),
        dense_shape=X.shape
    )

X_train_tf = scipy_to_tf_sparse(X_train)
X_valid_tf = scipy_to_tf_sparse(X_valid)

In [15]:
history = model.fit(
    X_train_tf,
    y_train,
    validation_data=(X_valid_tf, y_valid),
    epochs=3,
    batch_size=4096,
    verbose=1
)

Epoch 1/3


I0000 00:00:1782235603.729507      71 service.cc:152] XLA service 0x7e204c00c0e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782235603.729555      71 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1782235603.729561      71 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1782235603.993068      71 cuda_dnn.cc:529] Loaded cuDNN version 91002


  1/344 ━━━━━━━━━━━━━━━━━━━━ 1:01:09 11s/step - loss: 9.4619

I0000 00:00:1782235612.953895      71 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


344/344 ━━━━━━━━━━━━━━━━━━━━ 335s 946ms/step - loss: 0.4051 - val_loss: 0.1923
Epoch 2/3
344/344 ━━━━━━━━━━━━━━━━━━━━ 313s 909ms/step - loss: 0.1602 - val_loss: 0.1802
Epoch 3/3
344/344 ━━━━━━━━━━━━━━━━━━━━ 306s 888ms/step - loss: 0.1246 - val_loss: 0.1823


## Evaluation 

In [16]:
from sklearn.metrics import mean_squared_log_error

pred = model.predict(X_valid_tf, batch_size=4096).flatten()

rmsle = np.sqrt(
    mean_squared_log_error(
        np.expm1(y_valid),
        np.expm1(pred)
    )
)

print("RMSLE:", rmsle)

19/19 ━━━━━━━━━━━━━━━━━━━━ 5s 275ms/step
RMSLE: 0.4269459383830613


### Ignore the Code Below this

In [17]:
df.isnull().sum()
df.isnull().mean()*100

train_id              0.000000
name                  0.000000
item_condition_id     0.000000
category_name         0.426769
brand_name           42.675687
price                 0.000000
shipping              0.000000
item_description      0.000405
dtype: float64

In [18]:
df['name_len']=df['name'].str.len()
df['desc_len']=df['item_description'].str.len()
df['name_len']=df['item_description'].str.split().str.len()

##3 longer description mean more valuablle item(a note for me also ;/)


In [19]:
## now increase the input singnals 

df['has_brand'] = df['brand_name'].notnull().astype(int)
df['brand_name'] = df['brand_name'].fillna("unknown")


In [20]:
## validation of above logic 

df.groupby('has_brand')['price'].median()

has_brand
0    14.0
1    20.0
Name: price, dtype: float64

In [21]:
df.groupby('cat_main')['price'].median().sort_values(ascending = False)

KeyError: 'cat_main'

In [ ]:
df.groupby('cat_sub')['price'].median().sort_values(ascending = False).tail(15)

In [ ]:
df.groupby('cat_detail')['price'].median().sort_values(ascending = False).head(15)

In [ ]:

## checking cardinaltiy

print(df['cat_main'].nunique())  ### 10 OHE
print(df['cat_sub'].nunique())  ### Target Encoding replacing with mean
print(df['cat_detail'].nunique()) ### Target Encoding replacing mean

In [ ]:
### what is this standard

df[df['cat_detail']=='Standard'][['name','price','cat_main']].head(10)

In [ ]:
## target map

cat_sub_map = df.groupby('cat_sub')['price'].mean()
cat_detail_map = df.groupby('cat_detail')['price'].mean()

df['cat_sub_encoded'] = df['cat_sub'].map(cat_sub_map)
df['cat_detail_encoded'] = df['cat_detail'].map(cat_detail_map)


In [ ]:
df.drop(columns=['cat_sub_encoded', 'cat_detail_encoded'], inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=['price', 'log_price'])
y = df['log_price']

X_train, X_test, y_train, y_test = train_test_split(
 X, y, test_size=0.2, random_state=42
)


In [ ]:
X_train = pd.get_dummies(X_train, columns=['cat_main'], drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=['cat_main'], drop_first=True)

# Align columns (test might be missing some categories train has, or vice versa)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Verify
cat_main_cols = [col for col in X_train.columns if col.startswith('cat_main_')]
print(cat_main_cols)

In [ ]:
features = [
    'item_condition_id',
    'shipping',
    'has_brand',
    'cat_sub_encoded',
    'cat_detail_encoded',
    'name_len',
    'desc_len'
] + cat_main_cols

print(features)

# Verify all exist
for f in features:
    print(f, "->", f in X_train.columns)

In [ ]:
# You need y_train (log_price) to compute the encoding, not the dropped 'price' column
# Use expm1(y_train) to get back actual price, or just encode using log_price directly

cat_sub_map    = X_train.assign(log_price=y_train).groupby('cat_sub')['log_price'].mean()
cat_detail_map = X_train.assign(log_price=y_train).groupby('cat_detail')['log_price'].mean()

X_train['cat_sub_encoded']    = X_train['cat_sub'].map(cat_sub_map)
X_test['cat_sub_encoded']     = X_test['cat_sub'].map(cat_sub_map)

X_train['cat_detail_encoded']    = X_train['cat_detail'].map(cat_detail_map)
X_test['cat_detail_encoded']     = X_test['cat_detail'].map(cat_detail_map)

# Fill unseen categories in test with global mean
global_mean = y_train.mean()
X_train['cat_sub_encoded']    = X_train['cat_sub_encoded'].fillna(global_mean)
X_test['cat_sub_encoded']     = X_test['cat_sub_encoded'].fillna(global_mean)
X_train['cat_detail_encoded'] = X_train['cat_detail_encoded'].fillna(global_mean)
X_test['cat_detail_encoded']  = X_test['cat_detail_encoded'].fillna(global_mean)

In [ ]:
print(X_train.columns.tolist())

X_test

## For checking Purposes

In [ ]:
# Check NaN in encoded columns
print("NaN check:")
print("cat_sub_encoded NaN (train/test):", X_train['cat_sub_encoded'].isnull().sum(), X_test['cat_sub_encoded'].isnull().sum())
print("cat_detail_encoded NaN (train/test):", X_train['cat_detail_encoded'].isnull().sum(), X_test['cat_detail_encoded'].isnull().sum())

# Check y_train/y_test
print("\ny_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("y_train range:", y_train.min(), "-", y_train.max())

# Final check - features list complete and clean
print("\nFinal features list:")
print(features)
print("\nAll present:", all(f in X_train.columns for f in features))

In [ ]:
# Step 1: Build combined text (minimal cleaning)
X_train['text'] = (X_train['name'].fillna('') + ' ' + 
                    X_train['item_description'].fillna('')).str.lower().str.strip()

X_test['text'] = (X_test['name'].fillna('') + ' ' + 
                   X_test['item_description'].fillna('')).str.lower().str.strip()

print(X_train['text'].iloc[0])

## Embedding model 

In [ ]:
# Step 2: Load model
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model_st = SentenceTransformer('all-MiniLM-L6-v2', device=device)

model_st = model_st.to('cuda')
print(f"Embedding dimension: {model_st.get_sentence_embedding_dimension()}")

In [ ]:
# Step 3: Generate embeddings (faster batch size)
text_emb_train = model_st.encode(
    X_train['text'].tolist(), 
    batch_size=1024,
    show_progress_bar=True,
    convert_to_numpy=True
)

text_emb_test = model_st.encode(
    X_test['text'].tolist(), 
    batch_size=1024,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"Train embeddings shape: {text_emb_train.shape}")
print(f"Test embeddings shape: {text_emb_test.shape}")

In [ ]:
import numpy as np

X_train_num = X_train[features].fillna(0).astype(np.float32).values
X_test_num  = X_test[features].fillna(0).astype(np.float32).values

X_train_final = np.hstack([X_train_num, text_emb_train])
X_test_final  = np.hstack([X_test_num, text_emb_test])

print(f"Final train shape: {X_train_final.shape}")
print(f"Final test shape: {X_test_final.shape}")

## Model Training (Simple)


In [ ]:
import time
start = time.time()

model_lgb = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,
    random_state=42,
    device='gpu',
    n_jobs=-1,
    verbose=-1
)

model_lgb.fit(X_train_final, y_train)
print(f"Training time: {time.time()-start:.1f}s")

## Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

y_pred = model_lgb.predict(X_test_final)
y_pred = np.clip(y_pred, 0, 7.7)

rmsle = np.sqrt(mean_squared_error(y_test, y_pred))
rmse_dollar = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred)))

print(f"\n{'='*40}")
print(f"Sentence-Transformer + LightGBM (GPU)")
print(f"RMSLE:    {rmsle:.4f}")
print(f"RMSE ($): ${rmse_dollar:.2f}")
print(f"{'='*40}")

## MOdel Training

## Adding more features

## Aplllying TF-IDF

In [ ]:
=

In [ ]:
=

### MOdel